In [6]:
import os
import re
import pandas as pd
import numpy as np

from datasets import load_dataset
from sklearn.model_selection import train_test_split

from transformers import BertTokenizer

In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset("wangrongsheng/ag_news")

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [8]:
train_df = pd.DataFrame(dataset["train"])

train_df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [9]:
train_df["label"].value_counts()

label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64

In [10]:
L2_LABELS = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "SciTech"
}

L1_MAPPING = {
    0: 0,  # World -> Hard
    1: 1,  # Sports -> Soft
    2: 0,  # Business -> Hard
    3: 1   # SciTech -> Soft
}

L1_LABELS = {
    0: "Hard News",
    1: "Soft News"
}

In [11]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\\S+", "", text)

    text = re.sub(r"[^a-zA-Z0-9\\s]", "", text)

    text = re.sub(r"\\s+", " ", text)

    return text.strip()

In [12]:
def preprocess(example):

    cleaned_text = clean_text(example["text"])

    l2_label = example["label"]

    l1_label = L1_MAPPING[l2_label]

    return {
        "clean_text": cleaned_text,
        "l1_label": l1_label,
        "l2_label": l2_label
    }

In [13]:
processed_dataset = dataset.map(preprocess)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [14]:
train_valid = processed_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = train_valid["train"]

val_dataset = train_valid["test"]

test_dataset = processed_dataset["test"]

In [15]:
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\Govardhan\Documents\GitHub\NLP_S6\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Govardhan\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
def tokenize(batch):

    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [17]:
train_dataset = train_dataset.map(tokenize, batched=True)

val_dataset = val_dataset.map(tokenize, batched=True)

test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/108000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [18]:
columns = [
    "input_ids",
    "attention_mask",
    "l1_label",
    "l2_label"
]

train_dataset.set_format(
    type="torch",
    columns=columns
)

val_dataset.set_format(
    type="torch",
    columns=columns
)

test_dataset.set_format(
    type="torch",
    columns=columns
)

In [19]:
train_dataset.save_to_disk(
    "data/splits/train"
)

val_dataset.save_to_disk(
    "data/splits/val"
)

test_dataset.save_to_disk(
    "data/splits/test"
)

Saving the dataset (0/1 shards):   0%|          | 0/108000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7600 [00:00<?, ? examples/s]

In [22]:
import os

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/splits", exist_ok=True)

In [23]:
pd.DataFrame(train_dataset).to_csv(
    "data/processed/train.csv",
    index=False
)

pd.DataFrame(val_dataset).to_csv(
    "data/processed/val.csv",
    index=False
)

pd.DataFrame(test_dataset).to_csv(
    "data/processed/test.csv",
    index=False
)

In [24]:
from datasets import load_from_disk


def load_processed_data():

    train_dataset = load_from_disk(
        "data/splits/train"
    )

    val_dataset = load_from_disk(
        "data/splits/val"
    )

    test_dataset = load_from_disk(
        "data/splits/test"
    )

    return train_dataset, val_dataset, test_dataset

In [25]:
print(train_dataset[0])

{'l1_label': tensor(0), 'l2_label': tensor(0), 'input_ids': tensor([101, 100, 102,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0]), 'attention_mask': tensor([1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,